# 02 · GoldenCheetah OpenData audit (task 1.2)

Reads the index built by notebook 01 (metadata only, no ride samples) and answers:
how many riders and power rides there are, whether body weight is usable for W/kg, how many riders
have enough history, and whether **Gate 1** passes (≥ 500 riders with usable power history).

Works on the 100-athlete sample too, so you can run it before the full download finishes.
**At the end, send Claude the printed report.**

In [ ]:
# --- settings -------------------------------------------------------------
REPO = "Kevin-ai-jpg/BachelorThesis-MachineLearning-RoadCycling"
BRANCH = "main"
DRIVE_DATA = "/content/drive/MyDrive/thesis-cycling/data"
EXTRAS = "track"

In [ ]:
# Same setup as 00_colab_setup: Drive, repo, install, import
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess, pathlib, importlib
pathlib.Path(DRIVE_DATA).mkdir(parents=True, exist_ok=True)
os.environ["CYCLINGML_DATA"] = DRIVE_DATA
os.environ["CYCLINGML_MLRUNS"] = f"{DRIVE_DATA}/mlruns"
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
repo_dir = "/content/repo"
if os.path.isdir(f"{repo_dir}/.git"):
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", repo_dir, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, repo_dir], check=True)
%cd /content/repo
!git log -1 --oneline
!pip install -q -e ".[{EXTRAS}]"
if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")
importlib.invalidate_caches()
# Drop code imported earlier in this session, so the freshly pulled version is used
for _m in [m for m in sys.modules if m == "cyclingml" or m.startswith("cyclingml.")]:
    del sys.modules[_m]
from cyclingml import paths
GC = paths.raw_dir("goldencheetah")
GC.mkdir(parents=True, exist_ok=True)
print("GoldenCheetah folder:", GC)

In [ ]:
import pandas as pd
from cyclingml.config import load_config
from cyclingml.io.goldencheetah import load_index
from cyclingml.io.gc_audit import AuditConfig, run_audit, report_markdown

cfg_all = load_config("configs/audit_goldencheetah.yaml")
cfg = AuditConfig.from_dict(cfg_all["audit"])
athletes, rides = load_index(GC / "index")
print(f"{len(athletes)} athletes, {len(rides)} activities")
res = run_audit(athletes, rides, cfg)

## Riders

In [ ]:
riders = res["per_rider"]
riders.describe().round(1)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
riders["n_power_rides"].clip(upper=1000).plot.hist(bins=50, ax=ax[0], title="Power rides per rider (clipped at 1000)")
riders["span_days"].plot.hist(bins=50, ax=ax[1], title="Days between first and last power ride")
riders["mmp_20m_best"].plot.hist(bins=50, ax=ax[2], title="Best 20-min power (W)")
plt.tight_layout()

## Body weight

In [ ]:
from cyclingml.io.gc_audit import power_rides
pr = power_rides(rides)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
pr["weight_kg"].plot.hist(bins=100, range=(30, 150), ax=ax[0], title="Weight on power rides (kg)")
riders["weight_n_unique"].clip(upper=30).plot.hist(bins=30, ax=ax[1], title="Distinct weights per rider")
plt.tight_layout()
res["weight"]

## Report: send this to Claude

In [ ]:
report = report_markdown(res)
print(report)
(GC / "audit_report.md").write_text(report)
riders.to_parquet(GC / "index" / "riders.parquet")

In [ ]:
# Log the headline numbers to MLflow, so the audit is part of the experiment record
from cyclingml.tracking import run
with run(cfg_all.get("experiment", "data-audit"), cfg_all, run_name="goldencheetah-audit") as r:
    r.log_metrics({**{k: v for k, v in res["overview"].items() if isinstance(v, (int, float))},
                   "usable_riders": res["gate"]["usable_riders"],
                   "weight_plausible_share": res["weight"]["power_rides_weight_plausible"],
                   "weight_suspect_default_share": res["weight"]["power_rides_weight_suspect_default"]})
    r.log_artifact(str(GC / "audit_report.md"))
print("logged")